# 05 — Model Export and Deployment Preparation

## Objective
Verify the selected LightGBM log-target pipeline, validate its prediction interface, and prepare the saved model artifact for integration with the backend API.

The model is loaded from the trained artefact produced in Notebook 04. No model retraining is performed in this notebook.

In [2]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

MODEL_PATH = Path("../models/trained/lightgbm_log_target.joblib")
TEST_PATH = Path("../data/processed/test_2025.csv")

model = joblib.load(MODEL_PATH)
test_df = pd.read_csv(TEST_PATH)

# Reconstruct the same five input features used during training
FEATURES = [
    "property_type",
    "old_new",
    "duration",
    "district",
    "sale_month"
]

sample_property = test_df[FEATURES].iloc[[0]]

# Predict log price and convert back to pounds
predicted_log_price = model.predict(sample_property)[0]
predicted_price = np.exp(predicted_log_price)

print("Input features:")
display(sample_property)

print(f"Predicted property price: £{predicted_price:,.2f}")

Input features:


,property_type,old_new,duration,district,sale_month
0,F,N,L,BRENT,2


Predicted property price: £360,992.80


In [3]:
# Compare 10 known 2025 sale prices with model predictions

actual_prices = test_df["price"].iloc[:10].to_numpy()

sample_features = test_df[FEATURES].iloc[:10]

predicted_prices = np.exp(
    model.predict(sample_features)
)

prediction_check = pd.DataFrame({
    "Actual Price (£)": actual_prices,
    "Predicted Price (£)": predicted_prices.round(2),
    "Absolute Error (£)": np.abs(
        actual_prices - predicted_prices
    ).round(2),
    "Absolute Error (%)": (
        np.abs(actual_prices - predicted_prices)
        / actual_prices * 100
    ).round(2)
})

display(prediction_check)

,Actual Price (£),Predicted Price (£),Absolute Error (£),Absolute Error (%)
0,270000,360992.80,90992.80,33.70
1,840000,783235.58,56764.42,6.76
2,1420000,955143.33,464856.67,32.74
3,39601132,4836819.73,34764312.27,87.79
4,478000,891135.06,413135.06,86.43
5,495000,665511.66,170511.66,34.45
6,700000,1013313.73,313313.73,44.76
7,195000,926259.26,731259.26,375.00
8,181000,370333.77,189333.77,104.60
9,415000,936860.35,521860.35,125.75


In [4]:
# Inspect the features associated with the 10 predictions

error_details = test_df.loc[
    test_df.index[:10],
    FEATURES + ["price"]
].copy()

error_details["predicted_price"] = predicted_prices.round(2)

error_details["absolute_error_pct"] = (
    np.abs(
        error_details["price"].to_numpy() - predicted_prices
    )
    / error_details["price"].to_numpy()
    * 100
).round(2)

display(
    error_details.sort_values(
        "absolute_error_pct",
        ascending=False
    )
)

,property_type,old_new,duration,district,sale_month,price,predicted_price,absolute_error_pct
7,F,N,L,CITY OF WESTMINSTER,8,195000,926259.26,375.00
9,F,N,L,CITY OF WESTMINSTER,9,415000,936860.35,125.75
8,F,N,L,EALING,3,181000,370333.77,104.60
3,D,N,F,CAMDEN,9,39601132,4836819.73,87.79
4,F,N,L,CITY OF LONDON,9,478000,891135.06,86.43
6,F,N,L,CITY OF WESTMINSTER,3,700000,1013313.73,44.76
5,F,N,L,CAMDEN,9,495000,665511.66,34.45
0,F,N,L,BRENT,2,270000,360992.80,33.70
2,F,N,L,CITY OF WESTMINSTER,1,1420000,955143.33,32.74
1,T,N,F,BRENT,1,840000,783235.58,6.76
